In [1]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "gensim"])

0

In [2]:
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded!")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded!


In [3]:
dataset = load_dataset("mertbozkurt/llama2-TR-recipe")
df_raw  = pd.DataFrame(dataset['train'])

print("Shape:", df_raw.shape)
print("Sample:", df_raw['text'].iloc[0][:150])

Shape: (10504, 1)
Sample: <s>[INST] Sodalı Köfte için gerekli malzemeler nelerdir? [/INST] 500 gr kıyma, 1 adet büyük boy kuru soğan, 1/2 çay bardağıgaleta unu, 1 tatlı kaşığı 


In [5]:
recipes = {}

for row in df_raw['text']:
    malzeme_match = re.search(
        r'\[INST\](.+?)için gerekli malzemeler nelerdir\?\s*\[/INST\](.+?)(?:</s>|$)',
        row, re.DOTALL
    )
    if malzeme_match:
        tarif_adi  = malzeme_match.group(1).strip()
        malzemeler = malzeme_match.group(2).strip()
        if tarif_adi not in recipes:
            recipes[tarif_adi] = {}
        recipes[tarif_adi]['malzemeler'] = malzemeler

    yapilis_match = re.search(
        r'\[INST\](.+?)nasıl yapılır\?\s*\[/INST\](.+?)(?:</s>|$)',
        row, re.DOTALL
    )
    if yapilis_match:
        tarif_adi = yapilis_match.group(1).strip()
        yapilis   = yapilis_match.group(2).strip()
        if tarif_adi not in recipes:
            recipes[tarif_adi] = {}
        recipes[tarif_adi]['yapilis'] = yapilis

rows = []
for tarif_adi, info in recipes.items():
    if 'malzemeler' in info:
        rows.append({
            'recipe_name'   : tarif_adi,
            'ingredient_str': info.get('malzemeler', ''),
            'instructions'  : info.get('yapilis', ''),
        })

df = pd.DataFrame(rows)
print("Parsed recipes:", len(df))
df.head(3)

Parsed recipes: 3302


,recipe_name,ingredient_str,instructions
0,Sodalı Köfte,"500 gr kıyma, 1 adet büyük boy kuru soğan, 1/2...",Sodalı köfte yapımı için derin bir kap içine 1...
1,Hatay Kağıt Kebabı,"400 gr dana kıyma, 100 gr kuzu kıyma, 2 adet o...",Hatay kağıt kebabı yapımı için; öncelikle 2-3 ...
2,Piliç Topkapı,"4 adet kemiksiz tavuk but, 1 küçük çay bardağı...",Piliç topkapı yapımı için öncelikle kullanacağ...


In [6]:
EXCLUDE_KEYWORDS = [
    'kür', 'detoks', 'zayıflat', 'yağ yak', 'kilo ver',
    'nasıl saklanır', 'nasıl ayıklanır', 'nasıl közlenir',
    'nasıl kavrulur', 'nasıl mühürlenir', 'nasıl pişirilir',
    'benmari', 'buzlukta', 'dondurucuda',
    'metabolizma', 'ödem', 'öksürük', 'horlamayı',
    'yeşil kahve', 'çay kürü', 'sirkeli ballı'
]

KEEP_ANYWAY = [
    'Köfteli Kürdan Kebabı',
    'Çıngıllı Mantı Nasıl Yapılır',
    'Erişte Nasıl Yapılır',
    'Sıkma Mantı Nasıl Yapılır',
]

STOP_WORDS = {'tuz', 'su', 'sıcak su', 'kaynar su', 'ılık su', 'soğuk su'}

NORMALIZATION = {
    'sıvı yağ': ['sıvıyağ'],
    'kakao'   : ['silme kakao'],
    'soğan'   : ['kuru soğan'],
    'kıyma'   : ['köftelik kıyma'],
}
reverse_map = {v: k for k, vs in NORMALIZATION.items() for v in vs}

def is_valid_recipe(name):
    if name in KEEP_ANYWAY:
        return True
    return not any(kw in name.lower() for kw in EXCLUDE_KEYWORDS)

def clean_ingredient(ing):
    ing = ing.lower().strip()
    ing = re.sub(r'\(.*?\)', '', ing)
    ing = re.sub(r'\d+\s*/\s*\d+', '', ing)
    ing = re.sub(r'\d+[.,]?\d*', '', ing)
    ing = re.sub(r'(çay kaşığı|tatlı kaşığı|yemek kaşığı|çay bardağı|su bardağı)', ' ', ing)
    ing = re.sub(r'\b(gr|kg|ml|lt|cl|cc|adet|paket|demet|diş|tutam|dilim|dal|baş|sap|avuç|kutu|şişe|yaprak|top)\b', ' ', ing)
    ing = re.sub(r'\b(dolusu|kadar|büyük boy|orta boy|küçük boy|büyük|orta|küçük|ince|kalın|kuru|taze|rendelenmiş|doğranmış|haşlanmış|ezilmiş)\b', ' ', ing)
    ing = re.sub(r'^[\-–•*\s]+', '', ing)
    ing = re.sub(r'[/\\]', ' ', ing)
    ing = re.sub(r'\s+', ' ', ing).strip()
    return ing

def is_valid(ing):
    if ing in STOP_WORDS:
        return False
    if 'için' in ing or ':' in ing:
        return False
    return len(ing) > 1

def process_ingredients(ing_list):
    cleaned    = [clean_ingredient(i) for i in ing_list]
    normalized = [reverse_map.get(i, i) for i in cleaned]
    valid      = [i for i in normalized if is_valid(i)]
    seen, result = set(), []
    for i in valid:
        if i not in seen:
            seen.add(i)
            result.append(i)
    return result

# Uygula
df = df[df['recipe_name'].apply(is_valid_recipe)].reset_index(drop=True)
df['ingredients'] = df['ingredient_str'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip()])
df['ingredients_clean'] = df['ingredients'].apply(process_ingredients)
df['ingredient_count']  = df['ingredients_clean'].apply(len)

print(f"Recipes        : {len(df)}")
print(f"Avg ingredients: {df['ingredient_count'].mean():.1f}")
print(f"Sample         : {df['ingredients_clean'].iloc[0]}")

Recipes        : 3250
Avg ingredients: 7.9
Sample         : ['kıyma', 'soğan', 'galeta unu', 'kırmızı toz biber', 'kırmızı pul biber', 'kimyon', 'karabiber', 'kabartma tozu', 'soda']


In [7]:
sentences = df['ingredients_clean'].tolist()

w2v_model = Word2Vec(
    sentences  = sentences,
    vector_size= 100,
    window     = 5,
    min_count  = 1,
    workers    = 4,
    epochs     = 100
)

print("Word2Vec trained!")
print(f"Vocabulary size: {len(w2v_model.wv)}")
print()

# Benzer malzemeler test
print("=== 'tavuk' ile benzer ===")
for word, score in w2v_model.wv.most_similar('tavuk', topn=5):
    print(f"  {score:.3f}  {word}")

print()
print("=== 'kıyma' ile benzer ===")
for word, score in w2v_model.wv.most_similar('kıyma', topn=5):
    print(f"  {score:.3f}  {word}")

Word2Vec trained!
Vocabulary size: 1537

=== 'tavuk' ile benzer ===


KeyError: "Key 'tavuk' not present in vocabulary"

In [8]:
# Vocabulary'deki tavuk ile ilgili kelimeler
tavuk_words = [w for w in w2v_model.wv.key_to_index if 'tavuk' in w]
print("Tavuk related words:", tavuk_words)

print()

# Kıyma ile ilgili
kiyma_words = [w for w in w2v_model.wv.key_to_index if 'kıyma' in w]
print("Kıyma related words:", kiyma_words)

print()

# İlk 20 vocabulary
print("First 20 vocab:")
for w in list(w2v_model.wv.key_to_index.keys())[:20]:
    print(f"  {w}")

Tavuk related words: ['tavuk göğsü', 'tavuk suyu', 'kuşbaşı tavuk göğsü', 'tavuk baget', 'tavuk göğsü fileto', 'parça tavuk göğsü fileto', 'tavuk but', 'kuşbaşı tavuk eti', 'kemiksiz tavuk pirzola', 'kalçalı tavuk but', 'tavuk budu', 'tavuk fileto', 'kemiksiz tavuk incik', 'tavuk kanat', 'tavuk pirzola', 'bütün tavuk', 'tavuk butu', 'tavuk göğüsü', 'parça tavuk göğsü', 'tavuk ya da et suyu', 'tavuk eti', 'kemikli tavuk pirzola', 'tavuk kıyması', 'et suyu ya da tavuk suyu', 'tavuk göğüs eti', 'kemiksiz tavuk eti', 'kuşbaşı tavuk kalça eti', 'kemiksiz tavuk but', 'litre tavuk suyu', 'et ya da tavuk suyu', 'büyüklükte tavuk but', 'su ya da tavuk suyu', 'tavuk göğsü eti', 'köy tavuk budu', 'litre et suyu ya da tavuk suyu', 'kemikli tavuk göğsü', 'kuşbaşı tavuk göğüsü', 'kemiksiz fileto tavuk göğsü', 'tavuk suyu ya da et suyu', 'tavuk kalça', 'kilo kemiksiz tavuk but', 'parça kemiksiz tavuk incik', 'parça kemiksiz pirzola tavuk', 'dilimlenmiş tavuk fileto', 'kuşbaşı tavuk göğsü eti', 'kuşba

In [9]:
def get_recipe_vector(ingredients):
    vectors = []
    for ing in ingredients:
        if ing in w2v_model.wv:
            vectors.append(w2v_model.wv[ing])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(100)

# Her tarif için vektör oluştur
df['vector'] = df['ingredients_clean'].apply(get_recipe_vector)
recipe_vectors = np.stack(df['vector'].values)

print("Recipe vectors shape:", recipe_vectors.shape)
print()

# Kaç tarif vektörlenemedi
zero_vectors = (recipe_vectors.sum(axis=1) == 0).sum()
print(f"Recipes with no vector: {zero_vectors}")
print(f"Recipes with vector   : {len(df) - zero_vectors}")

Recipe vectors shape: (3250, 100)

Recipes with no vector: 0
Recipes with vector   : 3250


In [10]:
def recommend_w2v(user_ingredients, top_n=5):
    # Kullanıcı inputunu vektöre çevir
    user_vectors = []
    not_found = []
    
    for ing in user_ingredients:
        if ing in w2v_model.wv:
            user_vectors.append(w2v_model.wv[ing])
        else:
            not_found.append(ing)
    
    if not user_vectors:
        print("Hiçbir malzeme vocabulary'de bulunamadı!")
        return []
    
    if not_found:
        print(f"Vocabulary'de bulunamayan malzemeler: {not_found}")
    
    user_vector = np.mean(user_vectors, axis=0).reshape(1, -1)
    
    # Cosine similarity hesapla
    similarities = cosine_similarity(user_vector, recipe_vectors).flatten()
    top_indices  = similarities.argsort()[::-1]
    
    results = []
    for idx in top_indices[:top_n]:
        results.append({
            'recipe'           : df['recipe_name'].iloc[idx],
            'score'            : round(similarities[idx], 3),
            'ingredients_full' : df['ingredient_str'].iloc[idx].split(','),
            'instructions'     : df['instructions'].iloc[idx]
        })
    return results

# Test
print("=== Test: kıyma + soğan + domates ===")
for r in recommend_w2v(['kıyma', 'soğan', 'domates']):
    print(f"  {r['score']}  {r['recipe']}")

=== Test: kıyma + soğan + domates ===
  0.9330000281333923  Tortilladan Fındık Lahmacun
  0.9290000200271606  Lavaştan Lahmacun
  0.9129999876022339  Kıymalı Çiğ Börek
  0.902999997138977  Kıymalı Girit Kabağı Yemeği
  0.902999997138977  Kıymalı Bamya Yemeği


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [11]:
from sklearn.preprocessing import normalize

# Vektörleri normalize et
recipe_vectors_norm = normalize(recipe_vectors)

def recommend_w2v(user_ingredients, top_n=5):
    user_vectors = []
    not_found = []
    
    for ing in user_ingredients:
        if ing in w2v_model.wv:
            user_vectors.append(w2v_model.wv[ing])
        else:
            not_found.append(ing)
    
    if not user_vectors:
        print("Hiçbir malzeme vocabulary'de bulunamadı!")
        return []
    
    if not_found:
        print(f"Bulunamayan malzemeler: {not_found}")
    
    user_vector = np.mean(user_vectors, axis=0).reshape(1, -1)
    user_vector = normalize(user_vector)
    
    similarities = cosine_similarity(user_vector, recipe_vectors_norm).flatten()
    top_indices  = similarities.argsort()[::-1]
    
    results = []
    for idx in top_indices[:top_n]:
        results.append({
            'recipe'           : df['recipe_name'].iloc[idx],
            'score'            : round(float(similarities[idx]), 3),
            'ingredients_full' : df['ingredient_str'].iloc[idx].split(','),
            'instructions'     : df['instructions'].iloc[idx]
        })
    return results

# Test
print("=== Test: kıyma + soğan + domates ===")
for r in recommend_w2v(['kıyma', 'soğan', 'domates']):
    print(f"  {r['score']}  {r['recipe']}")

=== Test: kıyma + soğan + domates ===
  0.933  Tortilladan Fındık Lahmacun
  0.929  Lavaştan Lahmacun
  0.913  Kıymalı Çiğ Börek
  0.903  Kıymalı Girit Kabağı Yemeği
  0.903  Kıymalı Bamya Yemeği


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [12]:
import warnings
warnings.filterwarnings('ignore')

# Neden Tortilladan Fındık Lahmacun geliyor?
print("Tortilladan Fındık Lahmacun ingredients:")
idx = df[df['recipe_name'] == 'Tortilladan Fındık Lahmacun'].index[0]
print(df['ingredients_clean'].iloc[idx])
print()

# Lavaştan Lahmacun
print("Lavaştan Lahmacun ingredients:")
idx2 = df[df['recipe_name'] == 'Lavaştan Lahmacun'].index[0]
print(df['ingredients_clean'].iloc[idx2])

Tortilladan Fındık Lahmacun ingredients:
['tortilla', 'kıyma', 'soğan', 'domates', 'maydanoz', 'karabiber']

Lavaştan Lahmacun ingredients:
['lavaş', 'kıyma', 'soğan', 'domates', 'maydanoz', 'karabiber']


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF kur
df['ingredients_for_tfidf'] = df['ingredients_clean'].apply(lambda x: ' '.join(x))
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix     = tfidf_vectorizer.fit_transform(df['ingredients_for_tfidf'])

def recommend_tfidf(user_ingredients, top_n=5):
    user_str     = ' '.join(user_ingredients)
    user_vector  = tfidf_vectorizer.transform([user_str])
    similarities = cosine_similarity(user_vector, tfidf_matrix).flatten()
    top_indices  = similarities.argsort()[::-1]
    results = []
    for idx in top_indices[:top_n]:
        if similarities[idx] > 0:
            results.append({
                'recipe'          : df['recipe_name'].iloc[idx],
                'score'           : round(similarities[idx], 3),
                'ingredients_full': df['ingredient_str'].iloc[idx].split(','),
                'instructions'    : df['instructions'].iloc[idx]
            })
    return results

# Karşılaştır
user_input = ['kıyma', 'soğan', 'domates']

print("=== Word2Vec ===")
for r in recommend_w2v(user_input):
    print(f"  {r['score']}  {r['recipe']}")

print()
print("=== TF-IDF ===")
for r in recommend_tfidf(user_input):
    print(f"  {r['score']}  {r['recipe']}")

=== Word2Vec ===
  0.933  Tortilladan Fındık Lahmacun
  0.929  Lavaştan Lahmacun
  0.913  Kıymalı Çiğ Börek
  0.903  Kıymalı Girit Kabağı Yemeği
  0.903  Kıymalı Bamya Yemeği

=== TF-IDF ===
  0.855  Kıymalı Çiğ Börek
  0.703  Kıymalı Yumurta
  0.694  Kıyma Kavurması
  0.664  Fırında Kıymalı Patates Dizmesi
  0.664  Fırında Kıymalı Patates Yemeği


In [14]:
def recommend_hybrid(user_ingredients, top_n=5, w2v_weight=0.4, tfidf_weight=0.6):
    # Word2Vec scores
    user_vectors = [w2v_model.wv[i] for i in user_ingredients if i in w2v_model.wv]
    if user_vectors:
        user_vec_w2v = normalize(np.mean(user_vectors, axis=0).reshape(1, -1))
        scores_w2v   = cosine_similarity(user_vec_w2v, recipe_vectors_norm).flatten()
    else:
        scores_w2v = np.zeros(len(df))

    # TF-IDF scores
    user_vec_tfidf = tfidf_vectorizer.transform([' '.join(user_ingredients)])
    scores_tfidf   = cosine_similarity(user_vec_tfidf, tfidf_matrix).flatten()

    # Hybrid score
    hybrid_scores = w2v_weight * scores_w2v + tfidf_weight * scores_tfidf
    top_indices   = hybrid_scores.argsort()[::-1]

    results = []
    for idx in top_indices[:top_n]:
        results.append({
            'recipe'          : df['recipe_name'].iloc[idx],
            'score'           : round(float(hybrid_scores[idx]), 3),
            'ingredients_full': df['ingredient_str'].iloc[idx].split(','),
            'instructions'    : df['instructions'].iloc[idx]
        })
    return results

# Test
print("=== Hybrid (W2V %40 + TF-IDF %60) ===")
for r in recommend_hybrid(['kıyma', 'soğan', 'domates']):
    print(f"  {r['score']}  {r['recipe']}")

=== Hybrid (W2V %40 + TF-IDF %60) ===
  0.878  Kıymalı Çiğ Börek
  0.773  Kıymalı Yumurta
  0.761  Kıyma Kavurması
  0.751  Patates Oturtma
  0.751  Fırında Kıymalı Patates Dizmesi


In [15]:
test_cases = [
    (['yumurta', 'domates', 'biber', 'tereyağı'], 'Kahvaltı'),
    (['mercimek', 'soğan', 'havuç'],               'Çorba'),
    (['yufka', 'peynir', 'yumurta', 'tereyağı'],   'Börek'),
    (['kabak', 'yoğurt', 'sarımsak', 'dereotu'],   'Meze'),
    (['elma', 'tarçın', 'toz şeker', 'tereyağı'],  'Tatlı'),
]

for ingredients, category in test_cases:
    print(f"=== {category}: {ingredients} ===")
    for r in recommend_hybrid(ingredients, top_n=3):
        print(f"  {r['score']}  {r['recipe']}")
    print()

=== Kahvaltı: ['yumurta', 'domates', 'biber', 'tereyağı'] ===
  0.707  Soğansız Menemen
  0.698  Soğanlı Domatesli Yumurta
  0.672  Fransız Omleti

=== Çorba: ['mercimek', 'soğan', 'havuç'] ===
  0.741  Patatesli Mercimek Çorbası
  0.64  Sebzeli Yeşil Mercimek Yemeği
  0.638  Brokolili Mercimek Çorbası

=== Börek: ['yufka', 'peynir', 'yumurta', 'tereyağı'] ===
  0.755  Tek Yufka Böreği
  0.683  Patatesli Rulo Buzluk Böreği
  0.668  Patatesli Rulo Börek

=== Meze: ['kabak', 'yoğurt', 'sarımsak', 'dereotu'] ===
  0.962  Kabaklı Cacık
  0.868  Yoğurtlu Sebze Salatası
  0.833  Kabak Tarator

=== Tatlı: ['elma', 'tarçın', 'toz şeker', 'tereyağı'] ===
  0.841  Elmalı Crumble
  0.682  Ağızda Dağılan Elmalı Kurabiye
  0.678  Arası Elmalı Kek



In [16]:
test_cases = [
    (['yumurta', 'domates', 'biber', 'tereyağı'], 'Kahvaltı'),
    (['mercimek', 'soğan', 'havuç'],               'Çorba'),
    (['yufka', 'peynir', 'yumurta', 'tereyağı'],   'Börek'),
    (['kabak', 'yoğurt', 'sarımsak', 'dereotu'],   'Meze'),
    (['elma', 'tarçın', 'toz şeker', 'tereyağı'],  'Tatlı'),
    (['kıyma', 'soğan', 'domates'],                'Et'),
    (['tavuk göğsü', 'sarımsak', 'zeytinyağı'],    'Tavuk'),
    (['makarna', 'kıyma', 'domates'],              'Makarna'),
    (['patlıcan', 'domates', 'zeytinyağı'],         'Sebze'),
    (['un', 'yumurta', 'süt', 'toz şeker'],        'Hamur'),
]

for ingredients, category in test_cases:
    print(f"\n{'='*60}")
    print(f"Category : {category}")
    print(f"Input    : {ingredients}")
    print(f"{'='*60}")

    print("\n--- W2V ---")
    for r in recommend_w2v(ingredients, top_n=1):
        print(f"  Recipe: {r['recipe']} (Score: {r['score']})")
        for ing in r['ingredients_full']:
            print(f"    - {ing.strip()}")

    print("\n--- TF-IDF ---")
    for r in recommend_tfidf(ingredients, top_n=1):
        print(f"  Recipe: {r['recipe']} (Score: {r['score']})")
        for ing in r['ingredients_full']:
            print(f"    - {ing.strip()}")

    print("\n--- Hybrid ---")
    for r in recommend_hybrid(ingredients, top_n=1):
        print(f"  Recipe: {r['recipe']} (Score: {r['score']})")
        for ing in r['ingredients_full']:
            print(f"    - {ing.strip()}")


Category : Kahvaltı
Input    : ['yumurta', 'domates', 'biber', 'tereyağı']

--- W2V ---
  Recipe: Kuru Domatesli Sandviç (Score: 0.814)
    - 1 adet sandviç ekmek
    - 5 adet kuru domates
    - Tulum peyniri
    - Tereyağı

--- TF-IDF ---
  Recipe: Soğanlı Domatesli Yumurta (Score: 0.672)
    - 3 adet orta boy kuru soğan
    - 1 adet orta boy domates
    - 3 adet yumurta
    - 1 çay kaşığıdomates salçası
    - 1 tatlı kaşığı tereyağı
    - 3 yemek kaşığı zeytinyağı
    - Tuz

--- Hybrid ---
  Recipe: Soğansız Menemen (Score: 0.707)
    - 3 adet yumurta
    - 3 adet orta boy domates
    - 2 adet sivri biber
    - 1 tatlı kaşığı tereyağı
    - 1 yemek kaşığı zeytinyağı
    - Tuz

Category : Çorba
Input    : ['mercimek', 'soğan', 'havuç']

--- W2V ---
  Recipe: Portakallı Brüksel Lahanası (Score: 0.859)
    - 500 gr brüksel lahanası
    - 1 adet orta boy soğan
    - 1 adet orta boy havuç
    - 5 yemek kaşığı zeytinyağı
    - 1/2 su bardağı taze sıkılmış portakal suyu
    - 1/2 su bardağ

In [17]:
test_cases2 = [
    (['peynir', 'yumurta', 'sucuk'],                    'Kahvaltı 2'),
    (['pirinç', 'tavuk suyu', 'tereyağı'],               'Pilav'),
    (['ispanak', 'sarımsak', 'zeytinyağı', 'limon'],     'Ispanak'),
    (['çikolata', 'yumurta', 'tereyağı', 'toz şeker'],  'Çikolatalı Tatlı'),
    (['nohut', 'soğan', 'domates salçası', 'kimyon'],    'Nohut'),
    (['balık', 'limon', 'sarımsak', 'zeytinyağı'],       'Balık'),
    (['havuç', 'yoğurt', 'sarımsak', 'dereotu'],         'Meze 2'),
    (['bulgur', 'domates', 'biber salçası', 'soğan'],    'Bulgur'),
    (['patates', 'yumurta', 'soğan', 'zeytinyağı'],      'Patates'),
    (['kuru fasulye', 'soğan', 'domates salçası'],       'Kuru Fasulye'),
]

for ingredients, category in test_cases2:
    print(f"\n{'='*60}")
    print(f"Category : {category}")
    print(f"Input    : {ingredients}")
    print(f"{'='*60}")

    print("\n--- W2V ---")
    for r in recommend_w2v(ingredients, top_n=1):
        print(f"  Recipe: {r['recipe']} (Score: {r['score']})")
        for ing in r['ingredients_full']:
            print(f"    - {ing.strip()}")

    print("\n--- TF-IDF ---")
    for r in recommend_tfidf(ingredients, top_n=1):
        print(f"  Recipe: {r['recipe']} (Score: {r['score']})")
        for ing in r['ingredients_full']:
            print(f"    - {ing.strip()}")

    print("\n--- Hybrid ---")
    for r in recommend_hybrid(ingredients, top_n=1):
        print(f"  Recipe: {r['recipe']} (Score: {r['score']})")
        for ing in r['ingredients_full']:
            print(f"    - {ing.strip()}")


Category : Kahvaltı 2
Input    : ['peynir', 'yumurta', 'sucuk']

--- W2V ---
  Recipe: Fırında Sucuklu Kaşarlı Yumurta (Score: 0.953)
    - 2 adet yumurta
    - 10 dilim sucuk
    - 4 dilim kaşar peyniri
    - Kırımızı pul biber

--- TF-IDF ---
  Recipe: Sucuklu Omlet (Score: 0.834)
    - 2 adet yumurta
    - 5 küçük dilim sucuk
    - 2 yemek kaşığı rendelenmiş kaşar peynir
    - 1 tatlı kaşığı tereyağı
    - Tuz

--- Hybrid ---
  Recipe: Sucuklu Omlet (Score: 0.848)
    - 2 adet yumurta
    - 5 küçük dilim sucuk
    - 2 yemek kaşığı rendelenmiş kaşar peynir
    - 1 tatlı kaşığı tereyağı
    - Tuz

Category : Pilav
Input    : ['pirinç', 'tavuk suyu', 'tereyağı']

--- W2V ---
  Recipe: Şehriyeli Tavuk Çorbası (Score: 0.814)
    - 1 adet kalçalı tavuk but
    - 1/2 çay bardağı tel şehriye
    - 1
    - 5 yemek kaşığı tereyağı
    - 1 yemek kaşığı tepeleme un
    - 5 su bardağı tavuk suyu
    - Tuz

--- TF-IDF ---
  Recipe: Tavuklu Büryan Pilavı (Score: 0.677)
    - 1 su bardağı pilavlık

In [18]:
# CSV kaydet (DB Developer için)
df[['recipe_name', 'ingredient_str', 'ingredients_clean', 'instructions']].to_csv(
    'recipes_clean.csv',
    index=False,
    encoding='utf-8-sig'
)
print("CSV saved! Shape:", df.shape)

# PKL kaydet (Backend için)
import pickle

model = {
    'w2v_model'          : w2v_model,
    'tfidf_vectorizer'   : tfidf_vectorizer,
    'tfidf_matrix'       : tfidf_matrix,
    'recipe_vectors_norm': recipe_vectors_norm,
    'df'                 : df[['recipe_name', 'ingredients_clean', 'ingredient_str', 'instructions']]
}

with open('recommendation_model_hybrid.pkl', 'wb') as f:
    pickle.dump(model, f)

print("PKL saved!")
print(f"Recipes: {len(df)}")
print(f"W2V vocab: {len(w2v_model.wv)}")
print(f"TF-IDF vocab: {len(tfidf_vectorizer.vocabulary_)}")

CSV saved! Shape: (3250, 8)
PKL saved!
Recipes: 3250
W2V vocab: 1537
TF-IDF vocab: 776
